In [11]:
import pandas as pd
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Generators

from ax.core.observation import ObservationFeatures
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy
import json
import subprocess
import os
import re

In [12]:
iteration_to_update = 9
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"



In [16]:
ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update[trials_to_update['trial_index']>48]


,trial_index,arm_name,trial_status,generation_node,success,surfactant_input,complexity,Drug_MW,Drug_LogP,Drug_TPSA,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc
49,49,49_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.2063,0.3073,0.0373,0,0,0,0,100,0,100,74,100,100
50,50,50_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.4045,0.4196,0.0728,0,0,0,0,100,0,100,100,100,100
51,51,51_0,COMPLETED,GenerationStep_1,0.0,1.0,1.000,0.4045,0.4196,0.0728,29,0,0,0,100,0,100,100,100,100
52,52,52_0,COMPLETED,GenerationStep_1,0.0,1.0,1.000,0.2962,0.4364,0.0493,0,0,0,0,100,0,100,100,100,100
53,53,53_0,COMPLETED,GenerationStep_1,0.0,1.0,1.000,0.2962,0.4364,0.0493,44,100,0,0,100,100,85,100,100,100
54,54,54_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.3528,0.2810,0.0711,0,0,0,0,100,0,100,100,100,100
55,55,55_0,COMPLETED,GenerationStep_1,0.0,1.0,1.000,0.3528,0.2810,0.0711,100,100,0,100,0,100,0,100,1,100
56,56,56_0,COMPLETED,GenerationStep_1,1.0,1.0,0.625,0.2063,0.3073,0.0373,0,50,15,0,19,0,54,58,100,100
57,57,57_0,COMPLETED,GenerationStep_1,1.0,1.0,0.625,0.2063,0.3073,0.0373,0,32,0,0,15,79,56,84,100,100
58,58,58_0,COMPLETED,GenerationStep_1,1.0,1.0,0.500,0.4045,0.4196,0.0728,0,24,0,0,40,0,99,100,100,100


In [6]:
def update_data_to_optimizer(ax_client, list_of_new_failures):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    for trial_index in list_of_new_failures:
        # Make sure we actually have this trial
        if trial_index not in trials_df["trial_index"].values:
            print(f"Trial {trial_index} not found – skipping.")
            continue

        # Build the forced-failure payload
        new_data = {
            "success": 0,
            "surfactant_input": 1,
            "complexity": 1,
        }

        # Update the trial in-place
        ax_client.update_trial_data(trial_index=trial_index, raw_data=new_data)

    ax_client.save_to_json_file(updated_path)

    print(f"Updated trial {trial_index}: set success=0, surfactant_input=1, complexity=1")

    return ax_client

In [17]:
list_of_new_failures = [54,62,68,71]

In [18]:
print("Please double check the results you are updating...")
print("*" * 100)
print("*" * 100)

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


Please double check the results you are updating...
****************************************************************************************************
****************************************************************************************************


,trial_index,arm_name,trial_status,generation_node,success,surfactant_input,complexity,Drug_MW,Drug_LogP,Drug_TPSA,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc
54,54,54_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.3528,0.2810,0.0711,0,0,0,0,100,0,100,100,100,100
62,62,62_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.3528,0.2810,0.0711,0,0,0,0,50,0,56,100,100,100
68,68,68_0,COMPLETED,GenerationStep_1,1.0,1.0,0.500,0.2962,0.4364,0.0493,0,100,0,100,0,0,100,100,100,100
71,71,71_0,COMPLETED,GenerationStep_1,1.0,1.0,0.250,0.3528,0.2810,0.0711,0,0,0,0,0,0,100,100,100,100


In [19]:
new_ax_client = update_data_to_optimizer(ax_to_update, list_of_new_failures)

[INFO 07-16 09:42:35] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 54.
[INFO 07-16 09:42:35] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 62.
[INFO 07-16 09:42:35] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 68.
[INFO 07-16 09:42:35] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 71.


Updated trial 71: set success=0, surfactant_input=1, complexity=1


In [20]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials

,trial_index,arm_name,trial_status,generation_node,success,surfactant_input,complexity,Drug_MW,Drug_LogP,Drug_TPSA,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc
0,0,0_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,48,60,49,31,96,8,9,35,86,100
1,1,1_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,74,0,88,66,22,52,58,79,35,100
2,2,2_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,78,88,1,3,32,94,44,58,1,100
3,3,3_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,0,46,61,94,63,49,98,0,52,100
4,4,4_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2962,0.4364,0.0493,16,82,85,84,71,23,35,65,21,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,75,75_0,COMPLETED,GenerationStep_1,1.0,1.0,0.500,0.4045,0.4196,0.0728,0,0,0,100,100,0,100,100,100,100
76,76,76_0,COMPLETED,GenerationStep_1,1.0,1.0,0.125,0.2962,0.4364,0.0493,0,0,0,0,0,0,0,100,100,100
77,77,77_0,COMPLETED,GenerationStep_1,1.0,1.0,0.250,0.2962,0.4364,0.0493,0,0,0,100,0,0,0,100,100,100
78,78,78_0,COMPLETED,GenerationStep_1,1.0,1.0,0.250,0.3528,0.2810,0.0711,0,0,0,0,94,0,0,100,100,100
